# Phase 2 · LST agents: run a job

This notebook runs a downscaling job with Claude agents. The agents plan, start steps, review results and write the report. Earth Engine does the heavy work as batch tasks on Google's servers, and tested Python enforces every gate. Only you can approve a gate, and only in this notebook.

**How to use it:** Runtime → **Run all**, every time. The notebook stops on its own when it needs your approval or when Earth Engine is still busy, then disconnects the runtime to save compute units. Earth Engine keeps working while you're away. Come back later and Run all again; the run picks up where it left off.

**To approve a gate:** read the request printed by the advance cell (step 4). Set `GATE` in step 3 to the gate's name, then Run all again.

**Before the first run**

- Runtime → Change runtime type → **CPU**. Nothing here needs a GPU.
- Colab Secrets (the key icon), each with notebook access on: `LST_SA_KEY`, `ANTHROPIC_API_KEY` and `GITHUB_TOKEN`.
- In step 1, set `GITHUB_REPO` to your private repo.
- Start with `JOB_NAME = 'smoke_test'` in step 2. It runs a 16 km box with four passes and checks the whole pipeline against real Earth Engine for very little compute. Switch to `'san_antonio_p27'` when it finishes cleanly.
- Optional: put your NOAA LCD CSV for SAT in `gs://my-lst-project-lst/lst-agents/stations/` so QA can compare with air temperature.

**Rules this notebook keeps:** it never mounts Drive and never signs in as you. Don't add `drive.mount()`, `ee.Authenticate()` or `auth.authenticate_user()` to it.

## 1 · Setup

In [ ]:
# ── Your project ─────────────────────────────────────────────────────────────
PROJECT     = 'project-ec9a437d-ade6-43c0-b02'      # Cloud project ID
BUCKET      = 'my-lst-project-lst'                  # bucket name, without gs://
GITHUB_REPO = 'YOUR-GITHUB-USERNAME/lst-agents'     # ← your private repo
GITHUB_REF  = 'main'

RUN_FENCE_TEST       = True    # the containment checks; set False once they have passed
DISCONNECT_WHEN_DONE = True    # the last cell frees the runtime so it stops using compute units

import base64, os, subprocess, sys
if os.path.ismount('/content/drive') or os.path.exists('/content/drive/MyDrive'):
    raise SystemExit('Drive is mounted. Runtime → Disconnect and delete runtime; never mount Drive here.')
if 'YOUR-GITHUB-USERNAME' in GITHUB_REPO:
    raise SystemExit('Set GITHUB_REPO to your private repo (owner/name) first.')

# Pinned versions, so a new release can't slip in unnoticed
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'claude-agent-sdk==0.2.172',
                'earthengine-api==1.7.50'], check=True)

# Clone the code with the read-only token; the token is never written to disk or printed
REPO_DIR = '/content/lst-agents'
if not os.path.exists(REPO_DIR):
    from google.colab import userdata
    _tok = userdata.get('GITHUB_TOKEN')
    _auth = base64.b64encode(f'x-access-token:{_tok}'.encode()).decode()
    _env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0', 'GIT_CONFIG_COUNT': '1',
            'GIT_CONFIG_KEY_0': 'http.extraHeader', 'GIT_CONFIG_VALUE_0': f'Authorization: Basic {_auth}'}
    _r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_REF,
                         f'https://github.com/{GITHUB_REPO}.git', REPO_DIR], env=_env, capture_output=True, text=True)
    _err = _r.stderr.replace(_tok, '***')
    del _tok, _auth, _env
    if _r.returncode:
        raise SystemExit('Could not clone the repo. Check GITHUB_REPO and that the token can read it.\n' + _err[-400:])
print('Code:', subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s (%cd)'],
                              capture_output=True, text=True).stdout.strip())
sys.path.insert(0, REPO_DIR)

import lst_agents as lst
from lst_agents import approve, reject, status        # short names for the cells below
S = lst.connect(PROJECT, BUCKET)

In [ ]:
# ── Containment checks (once) ───────────────────────────────────────────────
# Checks that the fences refuse paths outside the project, that the hook refuses shell, file, web and
# other-agent tools, and that every refusal is logged. The live part runs one tiny agent session (about $0.01).
if RUN_FENCE_TEST:
    FENCES_OK = await lst.fence_test(S)
    if not FENCES_OK:
        raise SystemExit('Fix the failed checks before starting a run.')

## 2 · The job

`JOB_NAME` picks a job file from the repo's `jobs/` folder. To change a setting, edit that file on GitHub, then Runtime → Disconnect and delete runtime, and Run all. Or paste YAML into `JOB_OVERRIDE`. Settings you leave out take the defaults in `lst_agents/config.py`.

The cell starts a new run only if this job has no unfinished run, so running it again resumes the run you already have.

In [ ]:
JOB_NAME     = 'smoke_test'      # 'smoke_test' first, then 'san_antonio_p27'
JOB_OVERRIDE = ''                # optional: paste a whole job YAML here instead

JOB = JOB_OVERRIDE or open(f'{REPO_DIR}/jobs/{JOB_NAME}.yaml').read()
RUN_ID = lst.start_run(S, JOB)   # resumes this job's unfinished run if there is one
if not RUN_ID:
    raise SystemExit('Fix the job file (problems listed above).')

## 3 · Approve a gate

Leave `GATE` empty until the advance cell (step 4) asks for an approval. Read the request, then set `GATE` to its name and Run all again. The gates are `'plan'`, `'spend'`, `'model'` and `'qa'`. An approval is accepted only when that exact request is waiting.

To stop a run instead, run `reject(S, RUN_ID, 'plan', note='why')` in a new cell.

In [ ]:
GATE = ''            # e.g. 'plan', after you've read the request
NOTE = ''            # optional: why you approved; saved in the run record

if GATE:
    approve(S, RUN_ID, GATE, NOTE)

## 4 · Advance the run

This cell does everything that can happen next. It runs the agent session for the current stage, starts the Earth Engine tasks the agents asked for, and waits for those tasks. It moves on to the next stage until a gate needs you, the run is done, or `WAIT_MINUTES` pass. You can rerun it at any time.

In [ ]:
WAIT_MINUTES = 30    # how long to wait on Earth Engine before handing back (it keeps working either way)

RESULT = await lst.advance(S, RUN_ID, wait_minutes=WAIT_MINUTES)

## 5 · Status

In [ ]:
status(S, RUN_ID)

## 6 · Results

The maps appear once the run is done: mean anomaly, pass-to-pass variability, clear looks per pixel and trend over the whole area at 90 m, plus one 10 m close-up. The full report is `report.md` in the run folder.

In [ ]:
if RESULT == 'done':
    lst.show_results(S, RUN_ID)
    print(S.bucket.blob(S.fences.run_key(RUN_ID, 'report.md')).download_as_text())
else:
    print('Results appear here when the run is done.')

## 7 · Cleanup (yours, never an agent's)

After a run is done, its 90 m stacks are no longer needed. This lists them; nothing is deleted unless you set `DELETE_STACKS = True`.

In [ ]:
DELETE_STACKS = False

if RESULT == 'done':
    lst.cleanup(S, RUN_ID, 'stacks', confirm=DELETE_STACKS)

## 8 · Disconnect

In [ ]:
if DISCONNECT_WHEN_DONE:
    print(f'Run {RUN_ID}: {RESULT}. Disconnecting the runtime; the output above stays visible.')
    lst.finish()